# AItraceur — Entraînement CNN MobileNetV3-Small

Entraîne le scorer de postes CO sur les patches RG2 (238k) + Vikazimut (optionnel).
Produit `control_scorer_cnn.onnx` à copier dans `backend/data/models/`.

**Setup :**
1. Runtime → Modifier le type de runtime → GPU (T4)
2. Uploader le dataset sur Google Drive (voir Cellule 2)
3. Exécuter toutes les cellules dans l'ordre

In [ ]:
# ── Cellule 1 : Vérifier le GPU ─────────────────────────────────────────────
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Cellule 2 : Monter Google Drive ─────────────────────────────────────────
# Structure Drive attendue :
#   MyDrive/aitraceur/rg2/metadata.csv
#   MyDrive/aitraceur/rg2/train/pos/*.png
#   MyDrive/aitraceur/rg2/train/neg/*.png
#   (optionnel) MyDrive/aitraceur/vikazimut/metadata.csv
#   (optionnel) MyDrive/aitraceur/vikazimut/train/pos/*.png
from google.colab import drive
drive.mount('/content/drive')

import os
RG2_DIR      = '/content/drive/MyDrive/aitraceur/rg2'
VIKAZIMUT_DIR = '/content/drive/MyDrive/aitraceur/vikazimut'  # None si absent
OUTPUT_DIR   = '/content/drive/MyDrive/aitraceur/models'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Vérification
import pandas as pd
df = pd.read_csv(f'{RG2_DIR}/metadata.csv')
print(f'RG2 dataset : {len(df)} patches ({(df.label==1).sum()} pos, {(df.label==0).sum()} neg)')

if os.path.exists(f'{VIKAZIMUT_DIR}/metadata.csv'):
    df2 = pd.read_csv(f'{VIKAZIMUT_DIR}/metadata.csv')
    print(f'Vikazimut dataset : {len(df2)} patches ({(df2.label==1).sum()} pos, {(df2.label==0).sum()} neg)')
else:
    VIKAZIMUT_DIR = None
    print('Vikazimut dataset : absent (entraînement RG2 seul)')

In [ ]:
# ── Cellule 3 : Cloner le repo + installer dépendances ──────────────────────
!git clone https://github.com/guiguoz/AItraceur.git /content/AItraceur
%cd /content/AItraceur/backend
!pip install -q xgboost scikit-learn scipy pillow pandas numpy

In [ ]:
# ── Cellule 4 : Entraînement CNN ─────────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, 'scripts/train_control_scorer.py',
    '--phase', 'cnn',
    '--dataset-dir', RG2_DIR,
    '--output-dir', OUTPUT_DIR,
    '--epochs', '30',
    '--batch-size', '64',   # T4 peut gérer 64
]

if VIKAZIMUT_DIR:
    cmd += ['--extra-dataset-dir', VIKAZIMUT_DIR]

print('Commande :', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
print('Exit code :', result.returncode)

In [ ]:
# ── Cellule 5 : Vérifier le modèle exporté ───────────────────────────────────
import os
onnx_path = f'{OUTPUT_DIR}/control_scorer_cnn.onnx'
pth_path  = f'{OUTPUT_DIR}/best_model.pth'

for p in [onnx_path, pth_path]:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / 1e6
        print(f'✅ {os.path.basename(p)} — {size_mb:.1f} MB')
    else:
        print(f'❌ {p} — ABSENT')

In [ ]:
# ── Cellule 6 : Test inférence ONNX (vérification rapide) ────────────────────
!pip install -q onnxruntime

import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

# Batch de 4 patches aléatoires (224×224×3)
dummy = np.random.randn(4, 3, 224, 224).astype(np.float32)
logits = session.run(['output'], {'input': dummy})[0]
from scipy.special import expit
probs = expit(logits).ravel()

print('Inférence ONNX OK — 4 patches aléatoires :')
for i, p in enumerate(probs):
    print(f'  patch {i} → score = {p:.4f}')

## Après l'entraînement

1. Télécharger `control_scorer_cnn.onnx` depuis Google Drive
2. Copier dans `backend/data/models/control_scorer_cnn.onnx`
3. `pip install onnxruntime` dans l'env backend
4. Redémarrer uvicorn — le log affichera `CnnPatchScorer: chargé control_scorer_cnn.onnx`

**Aucun autre changement nécessaire** — fallback XGBoost automatique si .onnx absent.